# Fine-tune Cross-Encoder v0.6 - MSE + Spearman, 15 Epochs

Extended run to test if validation and test metrics continue improving beyond epoch 10.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` |
| **Evaluator** | Spearman correlation |
| **Epochs** | **15** (extended from 10) |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Goal**: Check if continuing training beyond epoch 10 yields better test LabelAcc.

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.5/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Helper Functions

In [ ]:
import json
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from typing import Any

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("✅ Helper functions loaded.")

## Load Dataset

In [ ]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

## Run 1: MSELoss + Spearman, 15 Epochs

In [ ]:
import torch
from torch.utils.data import DataLoader

run_name = "v0.6-mse-spearman-15ep"
output_dir = f"artifacts/models/cross-encoder-cv-jd-v0.6-mse-spearman-15ep"

model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Calculate warmup steps (10% of total steps)
total_steps = (len(train_examples) // 16 + 1) * 15
warmup_steps = int(total_steps * 0.1)

# Monkey-patch evaluator to print metrics after each epoch
original_call = evaluator.__call__
def evaluator_with_metrics(model, output_path=None, epoch=-1, steps=-1):
    result = original_call(model, output_path=output_path, epoch=epoch, steps=steps)
    if epoch >= 0:
        metrics = compute_metrics(model, val_examples)
        test_metrics = compute_metrics(model, test_examples)
        print(f"\n📊 Epoch {epoch + 1} completed:")
        print(f"  Validation: MAE={metrics['MAE']:.4f}, RMSE={metrics['RMSE']:.4f}, LabelAcc={metrics['LabelAcc']:.4f}")
        print(f"  Test:       MAE={test_metrics['MAE']:.4f}, RMSE={test_metrics['RMSE']:.4f}, LabelAcc={test_metrics['LabelAcc']:.4f}")
    return result

evaluator.__call__ = evaluator_with_metrics

print(f"🚀 Starting {run_name} (15 epochs)...")
print(f"⏱️  Total steps: {total_steps}, Warmup steps: {warmup_steps}\n")

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=15,
    loss_fct=torch.nn.MSELoss(),
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True,
    evaluation_steps=585  # Evaluate after each epoch
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\n🏆 Final Validation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\n🏆 Final Test metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

results = {
    'run': run_name,
    'loss': 'MSE',
    'evaluator': 'Spearman',
    'epochs': 15,
    'val': val_metrics,
    'test': test_metrics
}

## Save Report

In [ ]:
import os
import json
from pathlib import Path

# Create reports directory
os.makedirs('artifacts/reports', exist_ok=True)

# Save report
report = {
    'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'dataset_version': 'v0.5',
    'dataset_size': {
        'train': len(train_examples),
        'val': len(val_examples),
        'test': len(test_examples),
        'total': len(train_examples) + len(val_examples) + len(test_examples)
    },
    'run': results['run'],
    'loss': results['loss'],
    'evaluator': results['evaluator'],
    'epochs': results['epochs'],
    'metrics': {
        'validation': {k: float(v) for k, v in results['val'].items()},
        'test': {k: float(v) for k, v in results['test'].items()}
    },
    'model_path': 'artifacts/models/cross-encoder-cv-jd-v0.6-mse-spearman-15ep'
}

report_path = 'artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_15ep_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✅ Report saved to {report_path}")
print(f"\n📊 Summary:")
print(f"  Base model: cross-encoder/ms-marco-MiniLM-L-12-v2")
print(f"  Loss: {results['loss']}")
print(f"  Evaluator: {results['evaluator']}")
print(f"  Epochs: {results['epochs']}")
print(f"  Final Test LabelAcc: {results['test']['LabelAcc']:.4f} ({results['test']['LabelAcc']*100:.2f}%)")

## Save to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Save model
src_dir = 'artifacts/models/cross-encoder-cv-jd-v0.6-mse-spearman-15ep'
dest_dir = f"{drive_base}/models/cross-encoder-cv-jd-v0.6-mse-spearman-15ep"
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
shutil.copytree(src_dir, dest_dir)
print(f"✅ Model saved to Google Drive")

# Copy report
src = 'artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_15ep_report.json'
dest = f"{drive_base}/reports/fine_tune_cross_encoder_v0.6_mse_spearman_15ep_report.json"
shutil.copy(src, dest)
print(f"✅ Report saved to Google Drive")

print(f"\n✅ Done! Model and report saved to Google Drive!")